# Day 2: Ensemble Methods - Random Forests & Bagging

## Overview

Random Forests reduce tree overfitting by averaging predictions from multiple bootstrapped trees with random feature subsets. Master the ensemble paradigm that dominates real-world ML.

## Learning Objectives

- Understand bootstrap aggregating (bagging) principle
- Learn how random forests decorrelate trees
- Tune n_estimators, max_depth, min_samples_leaf
- Use Out-of-Bag (OOB) score for free validation

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report
import time

np.random.seed(42)
sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (12, 5)

## 1. Bootstrap Aggregating (Bagging)

In [ ]:
# Load and split data
iris = load_iris()
X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Single Decision Tree
dt = DecisionTreeClassifier(max_depth=10, random_state=42)
dt.fit(X_train, y_train)
dt_score = accuracy_score(y_test, dt.predict(X_test))

# Bagging (10 trees)
bag_10 = BaggingClassifier(DecisionTreeClassifier(max_depth=10), n_estimators=10, random_state=42)
bag_10.fit(X_train, y_train)
bag_10_score = accuracy_score(y_test, bag_10.predict(X_test))

# Bagging (100 trees)
bag_100 = BaggingClassifier(DecisionTreeClassifier(max_depth=10), n_estimators=100, random_state=42)
bag_100.fit(X_train, y_train)
bag_100_score = accuracy_score(y_test, bag_100.predict(X_test))

print(f"Single Tree Test Accuracy: {dt_score:.4f}")
print(f"Bagging (10 trees) Test Accuracy: {bag_10_score:.4f}")
print(f"Bagging (100 trees) Test Accuracy: {bag_100_score:.4f}")
print(f"\nImprovement: {(bag_100_score - dt_score):.4f}")

## 2. Random Forests: Adding Feature Randomness

In [ ]:
# Random Forest with different n_estimators
n_trees = [1, 5, 10, 20, 50, 100, 200]
rf_scores = []
train_times = []

for n in n_trees:
    start = time.time()
    rf = RandomForestClassifier(n_estimators=n, max_depth=10, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    train_times.append(time.time() - start)
    rf_scores.append(accuracy_score(y_test, rf.predict(X_test)))

# Plot: Accuracy vs n_estimators
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(n_trees, rf_scores, marker='o', linewidth=2, markersize=8)
ax1.set_xlabel('n_estimators')
ax1.set_ylabel('Test Accuracy')
ax1.set_title('Random Forest: Accuracy vs Number of Trees')
ax1.grid(alpha=0.3)

ax2.plot(n_trees, train_times, marker='s', color='orange', linewidth=2, markersize=8)
ax2.set_xlabel('n_estimators')
ax2.set_ylabel('Training Time (seconds)')
ax2.set_title('Random Forest: Training Time vs Number of Trees')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Best accuracy: {max(rf_scores):.4f} at {n_trees[np.argmax(rf_scores)]} trees")
print(f"Diminishing returns: accuracy plateaus after ~50 trees")

## 3. Feature Importance from Random Forests

In [ ]:
# Train Random Forest
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf.fit(X_train, y_train)

# Extract feature importance
feature_importance = rf.feature_importances_
sorted_idx = np.argsort(feature_importance)[::-1]

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(range(len(sorted_idx)), feature_importance[sorted_idx])
ax.set_yticks(range(len(sorted_idx)))
ax.set_yticklabels([iris.feature_names[i] for i in sorted_idx])
ax.set_xlabel('Mean Decrease Impurity')
ax.set_title('Random Forest Feature Importance')
plt.tight_layout()
plt.show()

# Model performance
train_acc = accuracy_score(y_train, rf.predict(X_train))
test_acc = accuracy_score(y_test, rf.predict(X_test))
print(f"Train Accuracy: {train_acc:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Generalization Gap: {(train_acc - test_acc):.4f}")

## 4. Out-of-Bag (OOB) Score

In [ ]:
# Train Random Forest with OOB scoring
rf_oob = RandomForestClassifier(n_estimators=100, oob_score=True, random_state=42)
rf_oob.fit(X_train, y_train)

# Get OOB score and test score
oob_score = rf_oob.oob_score_
test_score = accuracy_score(y_test, rf_oob.predict(X_test))

print(f"OOB Score: {oob_score:.4f}")
print(f"Test Score: {test_score:.4f}")
print(f"Difference: {abs(oob_score - test_score):.4f}")
print(f"\nOOB score is a good proxy for generalization without needing a separate validation set!")

## 5. Hyperparameter Tuning

In [ ]:
# Test different max_depth values
depths = range(3, 15)
rf_depth_scores = []

for depth in depths:
    rf = RandomForestClassifier(n_estimators=100, max_depth=depth, random_state=42)
    rf.fit(X_train, y_train)
    rf_depth_scores.append(accuracy_score(y_test, rf.predict(X_test)))

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(list(depths), rf_depth_scores, marker='o', linewidth=2, markersize=8)
ax.set_xlabel('max_depth')
ax.set_ylabel('Test Accuracy')
ax.set_title('Random Forest: Effect of max_depth')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

best_depth = list(depths)[np.argmax(rf_depth_scores)]
print(f"Best max_depth: {best_depth} (accuracy: {max(rf_depth_scores):.4f})")

## Exercises

1. Compare bagging with individual trees: is averaging always better?
2. Train a Random Forest with min_samples_leaf=1 vs min_samples_leaf=10. Which generalises better?
3. Use OOB score to estimate generalization error on Iris. Compare to cross-validation.

## Solutions

Key insights:
- Bagging reduces variance through averaging
- Random feature selection decorrelates trees
- n_estimators > 100 has diminishing returns
- OOB score ≈ k-fold CV with zero overhead